# Module 10 — Greedy Algorithms

This is the worked reference notebook: run live in lecture, fully solved.
The version students receive with TODOs in place of the solved parts is
`assignments/pds/a9-greedy/starter/greedy.py`.

## 1. Activity selection (Lecture 1)

In [1]:
def activity_selection(activities):
    activities = sorted(activities, key=lambda a: a[1])
    selected = [activities[0]]
    last_finish = activities[0][1]
    for start, finish in activities[1:]:
        if start >= last_finish:
            selected.append((start, finish))
            last_finish = finish
    return selected

activities = [(1,4),(3,5),(0,6),(5,7),(3,9),(5,9),(6,10),(8,11),(8,12),(2,14),(12,16)]
assert activity_selection(activities) == [(1,4),(5,7),(8,11),(12,16)]
print("Activity selection checks passed, matching Lecture 1's exact trace")

Activity selection checks passed, matching Lecture 1's exact trace


## 2. Shortest-duration-first counterexample, verified by brute force (Lecture 1 MTech)

In [2]:
def max_compatible(acts):
    best = 0
    n = len(acts)
    for mask in range(1 << n):
        sel = [acts[i] for i in range(n) if mask & (1 << i)]
        sel.sort(key=lambda a: a[1])
        ok = True
        for i in range(1, len(sel)):
            if sel[i][0] < sel[i-1][1]:
                ok = False; break
        if ok:
            best = max(best, len(sel))
    return best

def shortest_first(acts):
    acts = sorted(acts, key=lambda a: a[1] - a[0])
    sel = []
    for s, f in acts:
        if all(f <= s2 or s >= f2 for s2, f2 in sel):
            sel.append((s, f))
    return sel

counterexample = [(0,5),(4,6),(5,10)]
assert len(shortest_first(counterexample)) == 1   # shortest-first finds only 1
assert max_compatible(counterexample) == 2          # true optimum is 2
print("Shortest-duration-first counterexample confirmed: greedy gets 1, optimum is 2")

Shortest-duration-first counterexample confirmed: greedy gets 1, optimum is 2


## 3. Huffman coding (Lecture 2)

In [3]:
import heapq

def build_huffman_tree(freqs):
    heap = [[freq, [symbol, ""]] for symbol, freq in freqs.items()]
    heapq.heapify(heap)
    while len(heap) > 1:
        lo = heapq.heappop(heap)
        hi = heapq.heappop(heap)
        for pair in lo[1:]: pair[1] = '0' + pair[1]
        for pair in hi[1:]: pair[1] = '1' + pair[1]
        heapq.heappush(heap, [lo[0] + hi[0]] + lo[1:] + hi[1:])
    return sorted(heap[0][1:], key=lambda p: (len(p[-1]), p))

freqs = {"A": 45, "B": 13, "C": 12, "D": 16, "E": 14}
codes_list = build_huffman_tree(freqs)
codes = dict(codes_list)
assert codes == {"A": "0", "B": "101", "C": "100", "D": "111", "E": "110"}
total_bits = sum(freqs[c] * len(code) for c, code in codes.items())
assert total_bits == 210   # matches Lecture 2's exact trace

def decode(bits, codes):
    reverse = {v: k for k, v in codes.items()}
    result, current = [], ""
    for bit in bits:
        current += bit
        if current in reverse:
            result.append(reverse[current])
            current = ""
    return "".join(result)

message = "ABAC"
encoded = "".join(codes[ch] for ch in message)
assert encoded == "01010100"
assert decode(encoded, codes) == message
print("Huffman coding checks passed, matching Lecture 2's exact codes and trace")

Huffman coding checks passed, matching Lecture 2's exact codes and trace


## 4. Kruskal's algorithm for minimum spanning tree (Lecture 3)

In [4]:
def kruskal(n, edges):
    edges = sorted(edges)
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    mst = []
    for weight, u, v in edges:
        if find(u) != find(v):
            mst.append((u, v, weight))
            parent[find(u)] = find(v)
    return mst

edges = [(1,0,1),(2,1,2),(3,0,2),(4,2,3),(5,1,3)]
mst = kruskal(4, edges)
assert mst == [(0,1,1),(1,2,2),(2,3,4)]   # matches Lecture 3's exact trace
assert sum(w for _,_,w in mst) == 7
print("Kruskal's algorithm checks passed, matching Lecture 3's exact trace")

Kruskal's algorithm checks passed, matching Lecture 3's exact trace


## 5. Fractional vs. 0/1 knapsack (Lecture 4)

In [5]:
def fractional_knapsack(weights, values, W):
    items = sorted(range(len(weights)), key=lambda i: values[i] / weights[i], reverse=True)
    total_value = 0
    remaining = W
    for i in items:
        if weights[i] <= remaining:
            total_value += values[i]
            remaining -= weights[i]
        else:
            fraction = remaining / weights[i]
            total_value += values[i] * fraction
            remaining = 0
            break
    return total_value

weights, values, W = [10,20,30], [60,100,120], 50
assert fractional_knapsack(weights, values, W) == 240.0

def knapsack_01_bruteforce(weights, values, W):
    n = len(weights)
    best = 0
    for mask in range(1 << n):
        tw = sum(weights[i] for i in range(n) if mask & (1 << i))
        tv = sum(values[i] for i in range(n) if mask & (1 << i))
        if tw <= W:
            best = max(best, tv)
    return best

assert knapsack_01_bruteforce(weights, values, W) == 220
print("Fractional (240) vs 0/1 (220) knapsack checks passed, matching Lecture 4 exactly")

Fractional (240) vs 0/1 (220) knapsack checks passed, matching Lecture 4 exactly
